# [SOLUTION] Mini-Workshop 1 — Explore, Clean & Build a Baseline

**Dataset:** Melbourne housing (`melb_data.csv`) · **Time:** ~30 min · **Level:** beginner

Goal: take raw data all the way to a first prediction, using the **pipeline-free** style from the slides.

Steps: **Explore → Split → Clean → Train a baseline (Linear Regression)**

### Rules
- Fill in every line marked `# TODO`.
- Clean the data by fitting `SimpleImputer` / `StandardScaler` / `OneHotEncoder` **on the training set only**, then reuse them on the test set (no data snooping).
- Combine numeric + categorical arrays with `np.hstack` — no `Pipeline`, no `ColumnTransformer`.
- Keep `random_state=42` where asked.

## 1) Setup & load the data
Run this cell.

In [2]:
import os
import numpy as np
import pandas as pd

# Works in Google Colab (reads from the course repo) or locally (./datasets/)
URL = "https://raw.githubusercontent.com/NUS-ISS-SS/mla-day1-workshop-student/development/datasets/housing/melb_data.csv"
LOCAL = "./datasets/housing/melb_data.csv"
df = pd.read_csv(LOCAL) if os.path.exists(LOCAL) else pd.read_csv(URL)
print("Loaded Melbourne housing:", df.shape)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
df.head()

ModuleNotFoundError: No module named 'numpy'

## 2) Explore
Answer three questions in code: how big is the data, and which columns have missing values?

In [ ]:
# TODO: print the shape (rows, columns)
shape = df.shape  # <<
print("Shape:", shape)

In [ ]:
# TODO: build a table of missing values per column, sorted high -> low,
#       keeping only columns that actually have gaps.
missing = df.isnull().sum()  # <<
missing = missing[missing > 0].sort_values(ascending=False)  # <<
print(missing)

## 3) Choose features and separate numeric vs categorical
We give you a **curated feature list**. (We drop `Address`, `Suburb`, `SellerG`, `Date` on purpose — they have hundreds/thousands of unique text values and would blow up one-hot encoding.)

Your job: split these into numeric vs categorical columns with `select_dtypes`.

In [ ]:
SELECTED = ["Rooms","Distance","Postcode","Bedroom2","Bathroom","Car",
            "Landsize","BuildingArea","YearBuilt","Lattitude","Longtitude",
            "Propertycount","Type","Method","Regionname"]
data = df[SELECTED + ["Price"]].copy()
data.head()

In [ ]:
features = data.drop(columns=["Price"])
# TODO: numeric columns and categorical (object) columns of `features`
num_cols = list(features.select_dtypes(include=["number"]).columns)  # <<
cat_cols = list(features.select_dtypes(include=["object"]).columns)  # <<

assert set(num_cols).isdisjoint(cat_cols), "num and cat must not overlap"
assert set(num_cols) | set(cat_cols) == set(features.columns), "must cover all feature columns"
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

## 4) Define target `y` and features `X`, then split
80/20 split, `random_state=42`.

In [ ]:
# TODO: y is Price, X is everything else in `data`
y = data["Price"]  # <<
X = data.drop(columns=["Price"])  # <<

# TODO: train/test split (80/20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # <<

assert isinstance(y, pd.Series) and "Price" not in X.columns
assert len(X_train) + len(X_test) == len(X)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 5) Clean — the pipeline-free way
We clean the data in a strict order and keep each step separate:
1. **Fill missing values**
2. **Scale numeric columns**
3. **One-hot encode categorical columns**
4. **Combine** the processed numeric and categorical arrays with `np.hstack`

Each transformer uses `fit()` on the **training set only**, then `transform()` on both train and test.

In [ ]:
# Step 1a: fill missing values in the numeric columns
num_imputer = SimpleImputer(strategy="median")
num_imputer.fit(X_train[num_cols])

X_train_num_imputed = num_imputer.transform(X_train[num_cols])
X_test_num_imputed = num_imputer.transform(X_test[num_cols])

print("Numeric medians learned from training data:", np.round(num_imputer.statistics_, 1))

In [ ]:
# Step 1b: fill missing values in the categorical columns
cat_imputer = SimpleImputer(strategy="most_frequent")
cat_imputer.fit(X_train[cat_cols])

X_train_cat_imputed = cat_imputer.transform(X_train[cat_cols])
X_test_cat_imputed = cat_imputer.transform(X_test[cat_cols])

print("Categorical columns filled:", cat_cols)

### Step 2) Scale the numeric columns
Fit the scaler on the imputed numeric training data, then transform both train and test numeric arrays.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_num_imputed)

X_train_num_scaled = scaler.transform(X_train_num_imputed)
X_test_num_scaled = scaler.transform(X_test_num_imputed)

print("Scaled numeric matrix:", X_train_num_scaled.shape)

In [ ]:
# Step 3: one-hot encode the categorical columns
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(X_train_cat_imputed)

X_train_cat_encoded = encoder.transform(X_train_cat_imputed)
X_test_cat_encoded = encoder.transform(X_test_cat_imputed)

print("Encoded categorical matrix:", X_train_cat_encoded.shape)

### Step 4) Combine the processed arrays
Join the scaled numeric array and encoded categorical array side by side with `np.hstack`.

In [ ]:
# Step 4: combine the processed numeric and categorical arrays
X_train_prepared = np.hstack([X_train_num_scaled, X_train_cat_encoded])
X_test_prepared = np.hstack([X_test_num_scaled, X_test_cat_encoded])

assert X_train_prepared.shape[0] == len(X_train)
assert X_test_prepared.shape[0] == len(X_test)
assert X_train_prepared.shape[1] == X_test_prepared.shape[1]

print("Prepared training matrix:", X_train_prepared.shape)

## 6) Baseline model — Linear Regression
Train on the prepared training data, then measure test RMSE (in dollars).

In [ ]:
# TODO: train Linear Regression on the prepared training data
lin_reg = LinearRegression().fit(X_train_prepared, y_train)  # <<

# TODO: predict on the prepared TEST data
lin_pred = lin_reg.predict(X_test_prepared)  # <<

# TODO: test RMSE
lin_rmse = np.sqrt(mean_squared_error(y_test, lin_pred))  # <<

assert np.isfinite(lin_rmse) and lin_rmse > 0
print("Linear Regression Test RMSE: $", round(lin_rmse))

## 7) Reflect
Melbourne homes here have a median price near \$900k. Is a typical error of this size acceptable? Is this model **underfitting**? Keep that question — Mini-Workshop 2 introduces a model that behaves very differently.